# 4. Streamlit Demo App

This notebook installs the required packages and launches the interactive **Retail Recommendation Engine** demo built with [Streamlit](https://streamlit.io).

### What the app does
| Feature | Details |
|---------|---------|
| **Customer selector** | Choose a known B2B contractor (RFM-indexed) or switch to anonymous B2C mode |
| **Basket builder** | Search & select any product from the catalogue |
| **Hybrid recommendations** | Market Basket (Apriori) · Collaborative Filtering · Popularity fallback |
| **RFM profile card** | Recency / Frequency / Monetary / Segment displayed for B2B customers |
| **AI manager summary** | Paste an OpenAI API key to get a GPT-4o-mini plain-English explanation |

### Prerequisites
- Run **notebook 1** to generate `cleaned_retail_data_2022_2025.csv` and `rfm_base.csv`  
- Run **notebook 2** to generate `b2b_association_rules.csv`, `b2c_association_rules.csv`, and `item_similarity_matrix.pkl`

All files are expected in `data/preprocessed/` relative to this notebook.

## Step 1 — Install dependencies

In [ ]:
import subprocess, sys

packages = ["streamlit", "openai"]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "--quiet"],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(f"{pkg} is ready")
    else:
        print(f"Failed to install {pkg}")
        print(result.stderr)


## Step 2 — Verify data files exist

In [ ]:
import os

data_dir = os.path.join("data", "preprocessed")

required_files = [
    "cleaned_retail_data_2022_2025.csv",
    "rfm_base.csv",
    "b2b_association_rules.csv",
    "b2c_association_rules.csv",
]
optional_files = [
    "item_similarity_matrix.pkl",   # needed for CF recommendations (B2B)
]

all_ok = True
for fname in required_files:
    path = os.path.join(data_dir, fname)
    status = "OK" if os.path.exists(path) else "MISSING"
    if not os.path.exists(path):
        all_ok = False
    print(f"[{status}]  {path}")

for fname in optional_files:
    path = os.path.join(data_dir, fname)
    status = "OK" if os.path.exists(path) else "not found (run notebook 2 to enable CF recs)"
    print(f"[{status}]  {path}")

if all_ok:
    print("\nAll required files present - ready to launch the app!")
else:
    print("\nSome required files are missing. Run notebooks 1 & 2 first.")


## Step 3 — Quick smoke-test (no browser)

Loads data and runs the recommendation engine for one B2B and one B2C example 
to confirm everything works before launching the full app.

In [ ]:
import os
import pandas as pd

DATA_DIR = os.path.join("data", "preprocessed")

df          = pd.read_csv(os.path.join(DATA_DIR, "cleaned_retail_data_2022_2025.csv"))
rfm         = pd.read_csv(os.path.join(DATA_DIR, "rfm_base.csv"), index_col="CustomerName")
b2b_rules   = pd.read_csv(os.path.join(DATA_DIR, "b2b_association_rules.csv"))
b2c_rules   = pd.read_csv(os.path.join(DATA_DIR, "b2c_association_rules.csv"))

sim_path = os.path.join(DATA_DIR, "item_similarity_matrix.pkl")
item_similarity_df = pd.read_pickle(sim_path) if os.path.exists(sim_path) else pd.DataFrame()

popularity = (
    df.groupby("ProductName")["InvoiceID"]
    .nunique()
    .reset_index()
    .rename(columns={"InvoiceID": "PurchaseFrequency"})
    .sort_values("PurchaseFrequency", ascending=False)
)

print(f"Transactions : {len(df):,}")
print(f"Products     : {df['ProductName'].nunique():,}")
print(f"B2B customers: {len(rfm):,}")
print(f"B2B AR rules : {len(b2b_rules):,}")
print(f"B2C AR rules : {len(b2c_rules):,}")
print(f"CF matrix    : {'loaded' if not item_similarity_df.empty else 'not available'}")


In [ ]:
# ── Inline copy of the engine (mirrors app.py) ─────────────────────────────────
def get_hybrid_recommendations(customer_name, current_basket):
    is_b2c = customer_name == "SPRZEDAZ FISKALNA" or customer_name not in rfm.index

    rfm_profile = {}
    if not is_b2c and customer_name in rfm.index:
        r = rfm.loc[customer_name]
        rfm_profile = {
            "Recency":   int(r["Recency"]),
            "Frequency": int(r["Frequency"]),
            "Monetary":  round(float(r["Monetary"]), 2),
            "Segment":   str(r.get("SegmentLabel", "Unknown")),
        }

    result = {
        "Customer":          customer_name,
        "Type":              "B2C" if is_b2c else "B2B",
        "RFM_Profile":       rfm_profile,
        "CF_Recs":           [],
        "AR_Recs":           [],
        "Popularity_Fallback": [],
    }

    # CF
    if not is_b2c and not item_similarity_df.empty:
        cf_candidates = {}
        for item in current_basket:
            if item in item_similarity_df.columns:
                sims = item_similarity_df[item].sort_values(ascending=False).drop(item)
                for rec_item, score in sims.items():
                    if score > 0 and rec_item not in current_basket:
                        if rec_item not in cf_candidates or score > cf_candidates[rec_item]["Score"]:
                            cf_candidates[rec_item] = {"Item": rec_item, "Score": round(score, 2)}
        result["CF_Recs"] = sorted(cf_candidates.values(), key=lambda x: x["Score"], reverse=True)[:3]

    # AR
    rules_to_use = b2c_rules if is_b2c else b2b_rules
    ar_candidates = {}
    for item in current_basket:
        for _, row in rules_to_use[rules_to_use["antecedents"] == item].iterrows():
            rec_item = row["consequents"]
            if rec_item not in current_basket and rec_item not in ar_candidates:
                ar_candidates[rec_item] = {"Item": rec_item, "Lift": round(row["lift"], 2)}
    result["AR_Recs"] = sorted(ar_candidates.values(), key=lambda x: x["Lift"], reverse=True)[:3]

    # Fallback
    if not result["CF_Recs"] and not result["AR_Recs"]:
        result["Popularity_Fallback"] = [
            p for p in popularity["ProductName"] if p not in current_basket
        ][:3]

    return result


# ── B2B test ───────────────────────────────────────────────────────────────────
sample_customer = rfm.index[0]  # first known customer
sample_basket   = [df["ProductName"].dropna().iloc[0]]  # first product alphabetically

b2b_result = get_hybrid_recommendations(sample_customer, sample_basket)
print("── B2B test ──────────────────────────")
print(f"Customer  : {b2b_result['Customer']}")
print(f"Type      : {b2b_result['Type']}")
print(f"RFM       : {b2b_result['RFM_Profile']}")
print(f"AR  recs  : {[r['Item'] for r in b2b_result['AR_Recs']]}")
print(f"CF  recs  : {[r['Item'] for r in b2b_result['CF_Recs']]}")
print(f"Fallback  : {b2b_result['Popularity_Fallback']}")

# ── B2C test ───────────────────────────────────────────────────────────────────
b2c_result = get_hybrid_recommendations("SPRZEDAZ FISKALNA", sample_basket)
print("\n── B2C test ──────────────────────────")
print(f"Customer  : {b2c_result['Customer']}")
print(f"Type      : {b2c_result['Type']}")
print(f"AR  recs  : {[r['Item'] for r in b2c_result['AR_Recs']]}")
print(f"Fallback  : {b2c_result['Popularity_Fallback']}")

## Step 4 — Launch the app

In [ ]:
import subprocess, sys, os

app_path = os.path.join(os.getcwd(), "app.py")

# fmt: off
proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", app_path,
     "--server.headless", "true"],  # suppress automatic browser open from Streamlit
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
# fmt: on

print("Streamlit is starting… (this cell will keep running while the app is live)")
print("──────────────────────────────────────────────────────────────────")
for line in proc.stdout:
    print(line, end="")       # streams Streamlit's output into the cell
    if "Local URL" in line or "Network URL" in line:
        print("  ↑  Open the URL above in your browser")